# Old Permic OCR — CPU/CUDA Generation Benchmark

This is an optional fourth notebook. It preserves the original CPU renderer and adds a CUDA post-processing backend when PyTorch with CUDA is installed. The renderer itself remains the reference implementation because SVG parsing and exact YOLO geometry are CPU-owned; batched tensor colour/normalisation work is moved to the GPU.

Use `BACKEND='auto'` on Colab or a personal computer. It selects CUDA when available and safely falls back to CPU otherwise.


In [ ]:
# Cell 01 — setup
from pathlib import Path
import sys, time
REPO_DIR = Path('/content/ocroldpermic') if Path('/content/ocroldpermic').exists() else Path.cwd()
sys.path.insert(0, str(REPO_DIR / 'lib'))
from historical_glyph_curriculum.acceleration import detect_backend
from historical_glyph_curriculum.parallel.executor import CurriculumExecutor
print(detect_backend('auto'))


In [ ]:
# Cell 02 — optional GPU install for Colab
# Run only when needed; choose the wheel matching the runtime CUDA version.
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
# For local machines, install PyTorch from https://pytorch.org/get-started/locally/


In [ ]:
# Cell 03 — backend configuration
BACKEND = 'auto'       # 'auto', 'cpu', or 'cuda'
WORKERS = 'auto'        # CPU workers for SVG/material rendering
GPU_BATCH_SIZE = 64     # increase until VRAM is comfortably used
profile = detect_backend(BACKEND)
if BACKEND == 'cuda' and profile['backend'] != 'cuda':
    raise RuntimeError('CUDA was requested but is not available in this runtime')
print('Selected:', profile)


## Run a small stage

Use the same `GenerationPlan` and `CurriculumExecutor` as the first notebook. The only change is `backend='auto'` or `backend='cuda'`. The generated PNGs, labels, metadata, stage state, and GitHub checkpoint workflow remain compatible.


In [ ]:
# Cell 04 — construct the compatible executor
GLYPH_ROOT = REPO_DIR / 'font' / 'svg'
executor = CurriculumExecutor(
    glyph_root=GLYPH_ROOT, workers=WORKERS,
    backend=BACKEND, gpu_batch_size=GPU_BATCH_SIZE,
)
print('Executor backend:', executor.backend_info)


In [ ]:
# Cell 05 — benchmark saved-image post-processing without changing labels
from PIL import Image
import numpy as np
from historical_glyph_curriculum.acceleration import process_saved_images

# Point this to a completed stage or a small sample directory.
IMAGE_DIR = REPO_DIR / 'benchmark_images'
images = sorted(IMAGE_DIR.glob('*.png'))
if images:
    t0=time.perf_counter(); result=process_saved_images(images, backend=BACKEND, batch_size=GPU_BATCH_SIZE); elapsed=time.perf_counter()-t0
    print(result, f'elapsed={elapsed:.3f}s', f'images_per_second={len(images)/max(elapsed,1e-9):.2f}')
else:
    print('No benchmark_images/*.png found; run the first notebook or set IMAGE_DIR.')


## Operational notes

`auto` is the recommended setting. CPU remains the correctness and portability fallback. Multiple CPU workers should be reduced when GPU post-processing is enabled if host RAM is constrained. Keep `GPU_BATCH_SIZE` below the point where CUDA reports out-of-memory. The first notebook remains unchanged and can continue to produce and push stages to `colab-checkpoints`.
